In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/__results__.html
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/__huggingface_repos__.json
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/__notebook__.ipynb
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/__output__.json
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/custom.css
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm/train_results.json
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm/config.json
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm/trainer_state.json
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm/training_args.bin
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm/tokenizer.json
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm/all_results.json
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbe

# Init settings

In [2]:
from pathlib import Path
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# PROBE
PROBE_FILE = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "test-on-fabertwikifarsi-making-testset-sec6/"
    "factual_probe/factual_probe_dataset.csv"
)

# ParsBERT
PARSBERT_BASE_MODEL = "HooshvareLab/bert-base-parsbert-uncased"

#  ParsBERT-KG 
PARSBERT_KG_MODEL_PATH = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "8-finetunning-parsbert/parsbert_kg_mlm"
)

OUTPUT_DIR = Path("/kaggle/working/factual_probe_parsbert")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
MAX_LENGTH = 128
TOP_K_TO_SAVE = 10

print("Device:", DEVICE)
print("Probe file exists:", PROBE_FILE.exists())
print("ParsBERT KG model path exists:", PARSBERT_KG_MODEL_PATH.exists())
print("ParsBERT KG model path:", PARSBERT_KG_MODEL_PATH)


Device: cpu
Probe file exists: True
ParsBERT KG model path exists: True
ParsBERT KG model path: /kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm


In [3]:
probe_df = pd.read_csv(
    PROBE_FILE,
    encoding="utf-8-sig",
)

required_columns = {
    "fact_id",
    "subject",
    "predicate",
    "object",
    "prompt",
    "gold_answer",
    "object_token_count",
}

missing_columns = required_columns - set(probe_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

print("Initial shape:", probe_df.shape)
print("Columns:", probe_df.columns.tolist())

display(probe_df.head(10))

evaluation_df = probe_df.copy()

# this code just selects valid Columns
if "valid" in evaluation_df.columns:
    valid_values = (
        evaluation_df["valid"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
    )
    evaluation_df = evaluation_df[valid_values].copy()

evaluation_df = evaluation_df[
    evaluation_df["object_token_count"] == 1
].copy()

evaluation_df = evaluation_df.dropna(
    subset=["prompt", "gold_answer"]
).copy()

evaluation_df["prompt"] = (
    evaluation_df["prompt"]
    .astype(str)
    .str.strip()
)

evaluation_df["gold_answer"] = (
    evaluation_df["gold_answer"]
    .astype(str)
    .str.strip()
)

# preprocess 
evaluation_df = evaluation_df.drop_duplicates(
    subset=["fact_id", "prompt", "gold_answer"]
).reset_index(drop=True)

# check only one MASK
mask_counts = evaluation_df["prompt"].str.count(r"\[MASK\]")

invalid_mask_df = evaluation_df[
    mask_counts != 1
].copy()

evaluation_df = evaluation_df[
    mask_counts == 1
].reset_index(drop=True)

print("Final evaluation rows (before ParsBERT re-tokenization):", len(evaluation_df))
print("Rows rejected because mask count was not one:", len(invalid_mask_df))

print("\nPredicate distribution:")
display(
    evaluation_df["predicate"]
    .value_counts()
    .rename_axis("predicate")
    .reset_index(name="count")
)


Initial shape: (1206, 15)
Columns: ['fact_id', 'subject', 'predicate', 'object', 'prompt', 'template_id', 'template_type', 'gold_answer', 'gold_token_id', 'object_token_count', 'object_tokens', 'split_type', 'source', 'valid', 'validation_error']


,fact_id,subject,predicate,object,prompt,template_id,template_type,gold_answer,gold_token_id,object_token_count,object_tokens,split_type,source,valid,validation_error
0,fact_000001,محمد پورستار,محل تولد,اردبیل,زادگاه محمد پورستار [MASK] است.,birthplace_03,paraphrased,اردبیل,10050,1,"[""اردبیل""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
1,fact_000002,بخش ایرندگان,زبان,بلوچی,زبان بخش ایرندگان [MASK] است.,language_01,familiar,بلوچی,31449,1,"[""بلوچی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
2,fact_000003,پرم چوپرا,محل تولد,لاهور,محل تولد پرم چوپرا [MASK] است.,birthplace_01,familiar,لاهور,49461,1,"[""لاهور""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
3,fact_000004,یاروسلاو سایفرت,ملیت,چکی,یاروسلاو سایفرت فردی [MASK] است.,nationality_03,paraphrased,چکی,36644,1,"[""چکی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
4,fact_000005,سه برخوانی,کشور,تهران,کشور محل قرارگیری سه برخوانی، [MASK] است.,country_02,paraphrased,تهران,3148,1,"[""تهران""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
5,fact_000006,پاستورا سولر,ملیت,اسپانیایی,ملیت پاستورا سولر [MASK] است.,nationality_01,familiar,اسپانیایی,17013,1,"[""اسپانیایی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
6,fact_000007,اسکامپولو ۵۳ (فیلم ۱۹۵۳),زبان,ایتالیایی,زبان اسکامپولو ۵۳ (فیلم ۱۹۵۳) [MASK] است.,language_01,familiar,ایتالیایی,13857,1,"[""ایتالیایی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
7,fact_000008,قباد آذرآیین,محل تولد,مسجدسلیمان,محل تولد قباد آذرآیین [MASK] است.,birthplace_01,familiar,مسجدسلیمان,32192,1,"[""مسجدسلیمان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
8,fact_000009,ژرژ لامپن,ملیت,فرانسه,ژرژ لامپن فردی [MASK] است.,nationality_03,paraphrased,فرانسه,5916,1,"[""فرانسه""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
9,fact_000010,کشکش,استان,گیلان,کشکش در استان [MASK] قرار دارد.,province_01,familiar,گیلان,8188,1,"[""گیلان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN


Final evaluation rows (before ParsBERT re-tokenization): 1206
Rows rejected because mask count was not one: 0

Predicate distribution:


,predicate,count
0,ملیت,242
1,زبان,240
2,محل تولد,238
3,استان,233
4,کشور,208
5,زبان رسمی,45


- This time we don't rely on the old gold_token_id. Everything aligns from gold_answer and the ParsBERT tokenizer.

In [4]:
parsbert_tokenizer = AutoTokenizer.from_pretrained(
    PARSBERT_BASE_MODEL,
    use_fast=True,
)

print("ParsBERT tokenizer:", parsbert_tokenizer.name_or_path)
print("Vocabulary size:", len(parsbert_tokenizer))
print("Mask token:", parsbert_tokenizer.mask_token)
print("Mask token ID:", parsbert_tokenizer.mask_token_id)

tokenizer = parsbert_tokenizer  # برای استفاده در کل evaluation


config.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

ParsBERT tokenizer: HooshvareLab/bert-base-parsbert-uncased
Vocabulary size: 100000
Mask token: [MASK]
Mask token ID: 3


In [5]:
def tokenize_gold_answer(answer, tokenizer):
    token_ids = tokenizer.encode(
        str(answer),
        add_special_tokens=False,
    )
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    return token_ids, tokens


In [6]:
gold_token_counts = []
gold_token_ids = []
gold_tokens = []

for answer in evaluation_df["gold_answer"]:
    token_ids, tokens = tokenize_gold_answer(
        answer,
        tokenizer,
    )

    gold_token_counts.append(len(token_ids))
    gold_token_ids.append(
        token_ids[0] if len(token_ids) == 1 else None
    )
    gold_tokens.append(tokens)

evaluation_df["parsbert_gold_token_count"] = gold_token_counts
evaluation_df["parsbert_gold_token_id"] = gold_token_ids
evaluation_df["parsbert_gold_tokens"] = [
    json.dumps(tokens, ensure_ascii=False)
    for tokens in gold_tokens
]

new_multi_token_df = evaluation_df[
    evaluation_df["parsbert_gold_token_count"] != 1
].copy()

evaluation_df = evaluation_df[
    evaluation_df["parsbert_gold_token_count"] == 1
].reset_index(drop=True)

evaluation_df["parsbert_gold_token_id"] = (
    evaluation_df["parsbert_gold_token_id"]
    .astype(int)
)

print("Verified single-token rows for ParsBERT:", len(evaluation_df))
print("Rejected after ParsBERT token verification:", len(new_multi_token_df))

if len(new_multi_token_df):
    display(
        new_multi_token_df[
            [
                "gold_answer",
                "parsbert_gold_token_count",
                "parsbert_gold_tokens",
            ]
        ].head(20)
    )


Verified single-token rows for ParsBERT: 1205
Rejected after ParsBERT token verification: 1


,gold_answer,parsbert_gold_token_count,parsbert_gold_tokens
278,تورکی,2,"[""تورک"", ""##ی""]"


In [7]:
evaluation_df["verified_gold_token_id"] = evaluation_df["parsbert_gold_token_id"]
evaluation_df["verified_gold_token_count"] = evaluation_df["parsbert_gold_token_count"]
evaluation_df["verified_gold_tokens"] = evaluation_df["parsbert_gold_tokens"]

print("Final evaluation rows (ParsBERT-compatible):", len(evaluation_df))


Final evaluation rows (ParsBERT-compatible): 1205


# Evaluation function 

In [8]:
def clean_decoded_token(token):
    token = str(token)
    token = token.replace("##", "")
    token = token.replace("Ġ", "")
    token = token.replace("▁", "")
    token = token.strip()
    return token


@torch.inference_mode()
def evaluate_mlm_model(
    model,
    tokenizer,
    dataframe,
    model_name,
    batch_size=32,
    max_length=128,
    top_k_to_save=10,
):
    model.eval()

    mask_token_id = tokenizer.mask_token_id

    if mask_token_id is None:
        raise ValueError("Tokenizer does not have a mask token.")

    results = []
    total_inference_time = 0.0

    for start_index in tqdm(
        range(0, len(dataframe), batch_size),
        desc=f"Evaluating {model_name}",
    ):
        batch_df = dataframe.iloc[
            start_index:start_index + batch_size
        ].copy()

        prompts = batch_df["prompt"].tolist()

        encoded = tokenizer(
            prompts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        input_ids = encoded["input_ids"]

        mask_matrix = input_ids.eq(mask_token_id)
        mask_count_per_row = mask_matrix.sum(dim=1)

        if not torch.all(mask_count_per_row == 1):
            bad_rows = (
                mask_count_per_row.ne(1)
                .nonzero(as_tuple=False)
                .squeeze(-1)
                .detach()
                .cpu()
                .tolist()
            )

            bad_fact_ids = (
                batch_df.iloc[bad_rows]["fact_id"]
                .astype(str)
                .tolist()
            )

            raise ValueError(
                "Some tokenized prompts do not contain exactly one "
                f"mask token. Fact IDs: {bad_fact_ids[:20]}"
            )

        mask_positions = mask_matrix.long().argmax(dim=1)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        outputs = model(**encoded)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        total_inference_time += time.perf_counter() - start_time

        batch_indices = torch.arange(
            input_ids.size(0),
            device=DEVICE,
        )

        mask_logits = outputs.logits[
            batch_indices,
            mask_positions,
            :,
        ]

        log_probabilities = torch.log_softmax(
            mask_logits,
            dim=-1,
        )

        probabilities = torch.softmax(
            mask_logits,
            dim=-1,
        )

        gold_ids = torch.tensor(
            batch_df["verified_gold_token_id"].tolist(),
            dtype=torch.long,
            device=DEVICE,
        )

        gold_logits = mask_logits.gather(
            1,
            gold_ids.unsqueeze(1),
        ).squeeze(1)

        gold_log_probs = log_probabilities.gather(
            1,
            gold_ids.unsqueeze(1),
        ).squeeze(1)

        gold_probs = probabilities.gather(
            1,
            gold_ids.unsqueeze(1),
        ).squeeze(1)

        gold_ranks = (
            mask_logits > gold_logits.unsqueeze(1)
        ).sum(dim=1) + 1

        top_k = min(
            top_k_to_save,
            mask_logits.size(-1),
        )

        top_values, top_ids = torch.topk(
            log_probabilities,
            k=top_k,
            dim=-1,
        )

        top_probs = top_values.exp()

        gold_ids_cpu = gold_ids.detach().cpu().tolist()
        gold_ranks_cpu = gold_ranks.detach().cpu().tolist()
        gold_probs_cpu = gold_probs.detach().cpu().tolist()
        gold_log_probs_cpu = gold_log_probs.detach().cpu().tolist()
        top_ids_cpu = top_ids.detach().cpu().tolist()
        top_probs_cpu = top_probs.detach().cpu().tolist()

        for local_index, (_, source_row) in enumerate(
            batch_df.iterrows()
        ):
            predicted_ids = top_ids_cpu[local_index]

            predicted_tokens = tokenizer.convert_ids_to_tokens(
                predicted_ids
            )

            predicted_words = [
                clean_decoded_token(token)
                for token in predicted_tokens
            ]

            predicted_probabilities = top_probs_cpu[local_index]

            rank = int(gold_ranks_cpu[local_index])
            gold_id = int(gold_ids_cpu[local_index])

            top_predictions = [
                {
                    "rank": rank_index + 1,
                    "token": predicted_words[rank_index],
                    "token_id": int(predicted_ids[rank_index]),
                    "probability": float(
                        predicted_probabilities[rank_index]
                    ),
                }
                for rank_index in range(len(predicted_ids))
            ]

            result = source_row.to_dict()

            result.update({
                "model_name": model_name,
                "predicted_answer": predicted_words[0],
                "predicted_token_id": int(predicted_ids[0]),
                "predicted_probability": float(
                    predicted_probabilities[0]
                ),
                "gold_rank": rank,
                "gold_probability": float(
                    gold_probs_cpu[local_index]
                ),
                "gold_log_probability": float(
                    gold_log_probs_cpu[local_index]
                ),
                "reciprocal_rank": 1.0 / rank,
                "correct_at_1": int(rank <= 1),
                "correct_at_5": int(rank <= 5),
                "correct_at_10": int(rank <= 10),
                "top_predictions": json.dumps(
                    top_predictions,
                    ensure_ascii=False,
                ),
            })

            results.append(result)

        del outputs
        del mask_logits
        del log_probabilities
        del probabilities

    result_df = pd.DataFrame(results)

    speed_info = {
        "model_name": model_name,
        "evaluation_rows": len(result_df),
        "inference_seconds": total_inference_time,
        "examples_per_second": (
            len(result_df) / total_inference_time
            if total_inference_time > 0
            else np.nan
        ),
    }

    return result_df, speed_info


In [9]:
print("Loading baseline ParsBERT...")

parsbert_base_model = AutoModelForMaskedLM.from_pretrained(
    PARSBERT_BASE_MODEL,
)

parsbert_base_model.to(DEVICE)
parsbert_base_model.eval()

print("Baseline ParsBERT loaded.")
print("Model type:", parsbert_base_model.config.model_type)
print("Vocabulary size:", parsbert_base_model.config.vocab_size)
print(
    "Parameters:",
    f"{sum(p.numel() for p in parsbert_base_model.parameters()):,}",
)

baseline_results_parsbert, baseline_speed_parsbert = evaluate_mlm_model(
    model=parsbert_base_model,
    tokenizer=tokenizer,
    dataframe=evaluation_df,
    model_name="ParsBERT_baseline",
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    top_k_to_save=TOP_K_TO_SAVE,
)

print("Baseline ParsBERT evaluation completed.")
print(baseline_speed_parsbert)

display(
    baseline_results_parsbert[
        [
            "fact_id",
            "predicate",
            "prompt",
            "gold_answer",
            "predicted_answer",
            "gold_rank",
            "gold_probability",
            "correct_at_1",
            "correct_at_5",
            "correct_at_10",
        ]
    ].head(20)
)


Loading baseline ParsBERT...


pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/654M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: HooshvareLab/bert-base-parsbert-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Baseline ParsBERT loaded.
Model type: bert
Vocabulary size: 100000
Parameters: 239,842,880


Evaluating ParsBERT_baseline:   0%|          | 0/38 [00:00<?, ?it/s]

Baseline ParsBERT evaluation completed.
{'model_name': 'ParsBERT_baseline', 'evaluation_rows': 1205, 'inference_seconds': 61.63697407699999, 'examples_per_second': 19.549953871756482}


,fact_id,predicate,prompt,gold_answer,predicted_answer,gold_rank,gold_probability,correct_at_1,correct_at_5,correct_at_10
0,fact_000001,محل تولد,زادگاه محمد پورستار [MASK] است.,اردبیل,اصفهان,15,0.013519,0,0,0
1,fact_000002,زبان,زبان بخش ایرندگان [MASK] است.,بلوچی,فارسی,9,0.014513,0,0,1
2,fact_000003,محل تولد,محل تولد پرم چوپرا [MASK] است.,لاهور,هند,358,0.000298,0,0,0
3,fact_000004,ملیت,یاروسلاو سایفرت فردی [MASK] است.,چکی,لهستانی,1171,0.000058,0,0,0
4,fact_000005,کشور,کشور محل قرارگیری سه برخوانی، [MASK] است.,تهران,مالزی,146,0.000970,0,0,0
5,fact_000006,ملیت,ملیت پاستورا سولر [MASK] است.,اسپانیایی,فرانسوی,3,0.040115,0,1,1
6,fact_000007,زبان,زبان اسکامپولو ۵۳ (فیلم ۱۹۵۳) [MASK] است.,ایتالیایی,انگلیسی,29,0.001483,0,0,0
7,fact_000008,محل تولد,محل تولد قباد آذرآیین [MASK] است.,مسجدسلیمان,تهران,62,0.001738,0,0,0
8,fact_000009,ملیت,ژرژ لامپن فردی [MASK] است.,فرانسه,فرانسوی,116,0.000833,0,0,0
9,fact_000010,استان,کشکش در استان [MASK] قرار دارد.,گیلان,هرمزگان,4,0.057264,0,1,1


In [10]:
print("Loading KG-enhanced ParsBERT...")
print("Path:", PARSBERT_KG_MODEL_PATH)

parsbert_kg_model = AutoModelForMaskedLM.from_pretrained(
    str(PARSBERT_KG_MODEL_PATH),
)

parsbert_kg_model.to(DEVICE)
parsbert_kg_model.eval()

print("ParsBERT-KG model loaded.")
print("Model type:", parsbert_kg_model.config.model_type)
print("Vocabulary size:", parsbert_kg_model.config.vocab_size)
print(
    "Parameters:",
    f"{sum(p.numel() for p in parsbert_kg_model.parameters()):,}",
)

kg_results_parsbert, kg_speed_parsbert = evaluate_mlm_model(
    model=parsbert_kg_model,
    tokenizer=tokenizer,
    dataframe=evaluation_df,
    model_name="ParsBERT_KG",
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    top_k_to_save=TOP_K_TO_SAVE,
)

print("ParsBERT-KG evaluation completed.")
print(kg_speed_parsbert)

display(
    kg_results_parsbert[
        [
            "fact_id",
            "predicate",
            "prompt",
            "gold_answer",
            "predicted_answer",
            "gold_rank",
            "gold_probability",
            "correct_at_1",
            "correct_at_5",
            "correct_at_10",
        ]
    ].head(20)
)


Loading KG-enhanced ParsBERT...
Path: /kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


ParsBERT-KG model loaded.
Model type: bert
Vocabulary size: 100000
Parameters: 239,842,880


Evaluating ParsBERT_KG:   0%|          | 0/38 [00:00<?, ?it/s]

ParsBERT-KG evaluation completed.
{'model_name': 'ParsBERT_KG', 'evaluation_rows': 1205, 'inference_seconds': 73.19615503300076, 'examples_per_second': 16.4626133634577}


,fact_id,predicate,prompt,gold_answer,predicted_answer,gold_rank,gold_probability,correct_at_1,correct_at_5,correct_at_10
0,fact_000001,محل تولد,زادگاه محمد پورستار [MASK] است.,اردبیل,ی,96,0.001063,0,0,0
1,fact_000002,زبان,زبان بخش ایرندگان [MASK] است.,بلوچی,فارسی,5,0.010382,0,1,1
2,fact_000003,محل تولد,محل تولد پرم چوپرا [MASK] است.,لاهور,کابل,1161,0.000058,0,0,0
3,fact_000004,ملیت,یاروسلاو سایفرت فردی [MASK] است.,چکی,شخصیت,1321,0.000040,0,0,0
4,fact_000005,کشور,کشور محل قرارگیری سه برخوانی، [MASK] است.,تهران,ایران,9,0.001974,0,0,1
5,fact_000006,ملیت,ملیت پاستورا سولر [MASK] است.,اسپانیایی,فرانسوی,4,0.039207,0,1,1
6,fact_000007,زبان,زبان اسکامپولو ۵۳ (فیلم ۱۹۵۳) [MASK] است.,ایتالیایی,انگلیسی,4,0.015147,0,1,1
7,fact_000008,محل تولد,محل تولد قباد آذرآیین [MASK] است.,مسجدسلیمان,تهران,514,0.000129,0,0,0
8,fact_000009,ملیت,ژرژ لامپن فردی [MASK] است.,فرانسه,فرانسوی,5,0.002027,0,1,1
9,fact_000010,استان,کشکش در استان [MASK] قرار دارد.,گیلان,مازندران,2,0.231693,0,1,1


# Calculate metric

In [11]:
def calculate_metrics(result_df, model_name):
    ranks = result_df["gold_rank"].astype(float)

    return {
        "model": model_name,
        "samples": len(result_df),
        "P@1": result_df["correct_at_1"].mean(),
        "P@5": result_df["correct_at_5"].mean(),
        "P@10": result_df["correct_at_10"].mean(),
        "MRR": result_df["reciprocal_rank"].mean(),
        "Mean_Rank": ranks.mean(),
        "Median_Rank": ranks.median(),
        "Mean_Gold_Probability": result_df[
            "gold_probability"
        ].mean(),
        "Mean_Gold_LogProbability": result_df[
            "gold_log_probability"
        ].mean(),
    }


parsbert_metrics = pd.DataFrame([
    calculate_metrics(
        baseline_results_parsbert,
        "ParsBERT baseline",
    ),
    calculate_metrics(
        kg_results_parsbert,
        "ParsBERT KG",
    ),
])

metric_columns = [
    "P@1",
    "P@5",
    "P@10",
    "MRR",
    "Mean_Gold_Probability",
    "Mean_Gold_LogProbability",
]

for column in metric_columns:
    parsbert_metrics[column] = parsbert_metrics[column].astype(float)

display(parsbert_metrics.round(6))

percentage_metrics_parsbert = parsbert_metrics.copy()
for column in ["P@1", "P@5", "P@10"]:
    percentage_metrics_parsbert[column] = (
        percentage_metrics_parsbert[column] * 100
    ).round(2)

display(percentage_metrics_parsbert)


,model,samples,P@1,P@5,P@10,MRR,Mean_Rank,Median_Rank,Mean_Gold_Probability,Mean_Gold_LogProbability
0,ParsBERT baseline,1205,0.121162,0.285477,0.394191,0.206679,3549.068880,21.0,0.063844,-5.779687
1,ParsBERT KG,1205,0.118672,0.321162,0.464730,0.227483,2838.722822,13.0,0.088126,-5.578534


,model,samples,P@1,P@5,P@10,MRR,Mean_Rank,Median_Rank,Mean_Gold_Probability,Mean_Gold_LogProbability
0,ParsBERT baseline,1205,12.12,28.55,39.42,0.206679,3549.068880,21.0,0.063844,-5.779687
1,ParsBERT KG,1205,11.87,32.12,46.47,0.227483,2838.722822,13.0,0.088126,-5.578534


In [12]:
baseline_results_parsbert.to_csv(
    OUTPUT_DIR / "parsbert_baseline_factual_probe.csv",
    index=False,
    encoding="utf-8-sig",
)

kg_results_parsbert.to_csv(
    OUTPUT_DIR / "parsbert_kg_factual_probe.csv",
    index=False,
    encoding="utf-8-sig",
)

parsbert_metrics.to_csv(
    OUTPUT_DIR / "parsbert_factual_probe_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

percentage_metrics_parsbert.to_csv(
    OUTPUT_DIR / "parsbert_factual_probe_metrics_percent.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved ParsBERT factual probe results in:", OUTPUT_DIR)


Saved ParsBERT factual probe results in: /kaggle/working/factual_probe_parsbert
